# Loan Approval Prediction

**Models:**
- Logistic Regression (normal, Ridge, Lasso)
- Decision Tree Classifier (before and after grid search)
- Random Forest Classifier (before and after grid search)

**Dataset:** [Loan Approval Prediction Dataset (Kaggle)](https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset)



## 1. Load data

In [74]:
import pandas as pd
import numpy as np

df = pd.read_csv("loan_approval_dataset.csv")
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


## 2. Data Preprocessing

### 2.1 Explore the data

In [78]:
df.columns = df.columns.str.strip()          
print(df.shape)
df.info()

(4269, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4269 entries, 0 to 4268
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   loan_id                   4269 non-null   int64 
 1   no_of_dependents          4269 non-null   int64 
 2   education                 4269 non-null   object
 3   self_employed             4269 non-null   object
 4   income_annum              4269 non-null   int64 
 5   loan_amount               4269 non-null   int64 
 6   loan_term                 4269 non-null   int64 
 7   cibil_score               4269 non-null   int64 
 8   residential_assets_value  4269 non-null   int64 
 9   commercial_assets_value   4269 non-null   int64 
 10  luxury_assets_value       4269 non-null   int64 
 11  bank_asset_value          4269 non-null   int64 
 12  loan_status               4269 non-null   object
dtypes: int64(10), object(3)
memory usage: 433.7+ KB


In [80]:
df.describe()

,loan_id,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
count,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4.269000e+03,4.269000e+03
mean,2135.000000,2.498712,5.059124e+06,1.513345e+07,10.900445,599.936051,7.472617e+06,4.973155e+06,1.512631e+07,4.976692e+06
std,1232.498479,1.695910,2.806840e+06,9.043363e+06,5.709187,172.430401,6.503637e+06,4.388966e+06,9.103754e+06,3.250185e+06
min,1.000000,0.000000,2.000000e+05,3.000000e+05,2.000000,300.000000,-1.000000e+05,0.000000e+00,3.000000e+05,0.000000e+00
25%,1068.000000,1.000000,2.700000e+06,7.700000e+06,6.000000,453.000000,2.200000e+06,1.300000e+06,7.500000e+06,2.300000e+06
50%,2135.000000,3.000000,5.100000e+06,1.450000e+07,10.000000,600.000000,5.600000e+06,3.700000e+06,1.460000e+07,4.600000e+06
75%,3202.000000,4.000000,7.500000e+06,2.150000e+07,16.000000,748.000000,1.130000e+07,7.600000e+06,2.170000e+07,7.100000e+06
max,4269.000000,5.000000,9.900000e+06,3.950000e+07,20.000000,900.000000,2.910000e+07,1.940000e+07,3.920000e+07,1.470000e+07


In [82]:
df["loan_status"].value_counts()

loan_status
Approved    2656
Rejected    1613
Name: count, dtype: int64

### 2.2 Clean the data

- Strip spaces from text values
- Remove duplicates
- Drop `loan_id` (it is just an identifier)
- Handle missing values (numeric -> median, categorical -> mode)

In [85]:
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].str.strip()

df = df.drop_duplicates()
df = df.drop(columns=["loan_id"], errors="ignore")

print("Missing values before:", df.isnull().sum().sum())

num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include="object").columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

print("Missing values after:", df.isnull().sum().sum())

Missing values before: 0
Missing values after: 0


### 2.3 Encode categorical features

- Target: Approved = 1, Rejected = 0
- `education` and `self_employed` -> one-hot encoding

In [88]:
df["loan_status"] = df["loan_status"].map({"Approved": 1, "Rejected": 0})

cat_features = list(df.select_dtypes(include="object").columns)
df = pd.get_dummies(df, columns=cat_features, drop_first=True, dtype=int)
df.head()

,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status,education_Not Graduate,self_employed_Yes
0,2,9600000,29900000,12,778,2400000,17600000,22700000,8000000,1,0,0
1,0,4100000,12200000,8,417,2700000,2200000,8800000,3300000,0,1,1
2,3,9100000,29700000,20,506,7100000,4500000,33300000,12800000,0,0,0
3,3,8200000,30700000,8,467,18200000,3300000,23300000,7900000,0,0,0
4,5,9800000,24200000,20,382,12400000,8200000,29400000,5000000,0,1,1


### 2.4 Train / Test split (80 / 20, same as the example) and scaling

Scaling is applied for Logistic Regression only. Tree-based models do not need it.

In [91]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns="loan_status")
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (3415, 11) | Test: (854, 11)


## 3. Evaluation function

Returns Train and Test metrics (Accuracy, Precision, Recall, F1) for any fitted model.

In [94]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def eval_model(model_name, model, X_train, X_test, y_train, y_test):

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    results = {
        "Model": model_name,

        "Train Accuracy": accuracy_score(y_train, y_train_pred),
        "Train Recall": recall_score(y_train, y_train_pred, average="weighted", zero_division=0),
        "Train Precision": precision_score(y_train, y_train_pred, average="weighted", zero_division=0),
        "Train F1": f1_score(y_train, y_train_pred, average="weighted", zero_division=0),

        "Test Accuracy": accuracy_score(y_test, y_test_pred),
        "Test Precision": precision_score(y_test, y_test_pred, average="weighted", zero_division=0),
        "Test Recall": recall_score(y_test, y_test_pred, average="weighted", zero_division=0),
        "Test F1": f1_score(y_test, y_test_pred, average="weighted", zero_division=0),
    }

    return results

results = []

## 4. Logistic Regression

### 4.1 Normal (no regularization)

In [98]:
from sklearn.linear_model import LogisticRegression

model_LOG = LogisticRegression(penalty=None, max_iter=1000)
model_LOG.fit(X_train_s, y_train)

results.append(
    eval_model("Logistic Regression (normal)", model_LOG, X_train_s, X_test_s, y_train, y_test)
)

### 4.2 Ridge (L2)

In [101]:
model_RIDGE = LogisticRegression(penalty="l2", max_iter=1000)
model_RIDGE.fit(X_train_s, y_train)

results.append(
    eval_model("Logistic Regression (Ridge)", model_RIDGE, X_train_s, X_test_s, y_train, y_test)
)

### 4.3 Lasso (L1)

In [104]:
model_LASSO = LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000)
model_LASSO.fit(X_train_s, y_train)

results.append(
    eval_model("Logistic Regression (Lasso)", model_LASSO, X_train_s, X_test_s, y_train, y_test)
)

pd.Series(model_LASSO.coef_[0], index=X.columns).round(3)

no_of_dependents           -0.073
income_annum               -1.625
loan_amount                 1.366
loan_term                  -0.872
cibil_score                 4.161
residential_assets_value    0.038
commercial_assets_value     0.089
luxury_assets_value         0.121
bank_asset_value            0.175
education_Not Graduate     -0.063
self_employed_Yes           0.011
dtype: float64

## 5. Decision Tree Classifier

### 5.1 Before grid search

In [108]:
from sklearn.tree import DecisionTreeClassifier

model_DT = DecisionTreeClassifier(random_state=42)
model_DT.fit(X_train, y_train)

results.append(
    eval_model("DecisionTreeClassifier before Grid", model_DT, X_train, X_test, y_train, y_test)
)

### 5.2 After grid search

In [111]:
from sklearn.model_selection import GridSearchCV

param_grid_DT = {
    "max_depth": [2, 3, 4, 5, 6, 7, 8, 9, 10],
    "min_samples_split": [2, 5, 10],
}

grid_DT = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_DT, cv=5)
grid_DT.fit(X_train, y_train)

print("Best params:", grid_DT.best_params_)
print("Best CV score:", round(grid_DT.best_score_, 4))

results.append(
    eval_model("DecisionTreeClassifier after Grid", grid_DT.best_estimator_, X_train, X_test, y_train, y_test)
)

Best params: {'max_depth': 8, 'min_samples_split': 2}
Best CV score: 0.9739


## 6. Random Forest Classifier

### 6.1 Before grid search

In [114]:
from sklearn.ensemble import RandomForestClassifier

model_RF = RandomForestClassifier(random_state=42)
model_RF.fit(X_train, y_train)

results.append(
    eval_model("RandomForestClassifier before Grid", model_RF, X_train, X_test, y_train, y_test)
)

### 6.2 After grid search

In [116]:
param_grid_RF = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 8, 10, 15],
}

grid_RF = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_RF, cv=5, n_jobs=-1)
grid_RF.fit(X_train, y_train)

print("Best params:", grid_RF.best_params_)
print("Best CV score:", round(grid_RF.best_score_, 4))

results.append(
    eval_model("RandomForestClassifier after Grid", grid_RF.best_estimator_, X_train, X_test, y_train, y_test)
)

Best params: {'max_depth': 15, 'n_estimators': 150}
Best CV score: 0.9789


## 7. Final comparison (Train & Test data)

In [119]:
results_df = pd.DataFrame(results).round(4)
results_df

,Model,Train Accuracy,Train Recall,Train Precision,Train F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,Logistic Regression (normal),0.9168,0.9168,0.9167,0.9168,0.9110,0.9108,0.9110,0.9105
1,Logistic Regression (Ridge),0.9183,0.9183,0.9182,0.9183,0.9133,0.9131,0.9133,0.9130
2,Logistic Regression (Lasso),0.9192,0.9192,0.9191,0.9192,0.9133,0.9131,0.9133,0.9130
3,DecisionTreeClassifier before Grid,1.0000,1.0000,1.0000,1.0000,0.9836,0.9836,0.9836,0.9836
4,DecisionTreeClassifier after Grid,0.9909,0.9909,0.9910,0.9909,0.9789,0.9792,0.9789,0.9788
5,RandomForestClassifier before Grid,1.0000,1.0000,1.0000,1.0000,0.9813,0.9813,0.9813,0.9813
6,RandomForestClassifier after Grid,1.0000,1.0000,1.0000,1.0000,0.9824,0.9824,0.9824,0.9824


In [120]:
# Overfitting check: gap between Train F1 and Test F1
results_df["F1 Gap (Train - Test)"] = (results_df["Train F1"] - results_df["Test F1"]).round(4)
print("Best model on Test F1:", results_df.loc[results_df["Test F1"].idxmax(), "Model"])
results_df[["Model", "Train F1", "Test F1", "F1 Gap (Train - Test)"]]

Best model on Test F1: DecisionTreeClassifier before Grid


,Model,Train F1,Test F1,F1 Gap (Train - Test)
0,Logistic Regression (normal),0.9168,0.9105,0.0063
1,Logistic Regression (Ridge),0.9183,0.9130,0.0053
2,Logistic Regression (Lasso),0.9192,0.9130,0.0062
3,DecisionTreeClassifier before Grid,1.0000,0.9836,0.0164
4,DecisionTreeClassifier after Grid,0.9909,0.9788,0.0121
5,RandomForestClassifier before Grid,1.0000,0.9813,0.0187
6,RandomForestClassifier after Grid,1.0000,0.9824,0.0176
